# Communes social economical details

In [ ]:
import os
os.environ['PATH'] += ':/opt/hadoop/bin/:/opt/hadoop/sbin/'

import datetime
import pandas as pd
import numpy as np

import geopandas as gpd

from descartes.patch import PolygonPatch
import matplotlib.cm as cm
import matplotlib.colors as colrs

from pyspark.sql import SparkSession
from pyspark.sql import DataFrame
from pyspark.conf import SparkConf

from pyspark.sql import functions as spark_functions
from pyspark.sql import types as spark_types

from pyspark.sql.functions import pandas_udf

from pydoop.hdfs import hdfs
from functools import reduce

from tqdm import tqdm
from matplotlib import pyplot as plt
import folium
import json
import mapply
from geovoronoi import voronoi_regions_from_coords
from pyproj import Proj, transform
from matplotlib.path import Path
from shapely.geometry import Point, mapping
from shapely.geometry import shape as Shape
from shapely.ops import transform as Shapely_transform

import json
import folium

In [ ]:
def get_map(init_location=[46.2276, 2.2137], 
            zoom_start=10, 
            zoom_control=False,
            style='light'):

    styles = {
    'light': 'mapbox/light-v9',
    'dark': 'mapbox/dark-v9',
    'streets': 'mapbox/streets-v11',
    'light_landy': 'mapbox/clnm5177k004m01pae5n6c42u',
    }
    if style not in styles:
        raise ValueError('Style not supported')
    style = styles[style]

    m = folium.Map(location=init_location, zoom_control=zoom_control, zoom_start=zoom_start, 
                    tiles=f'https://api.mapbox.com/styles/v1/{style}/tiles/{{z}}/{{x}}/{{y}}?access_token=pk.eyJ1Ijoia3ViYS1yZWlzZW4iLCJhIjoiY2p5NjNkMjRhMGU3bDNkc3l1OWI3OXl0byJ9.m5V748nSYU5KWdQ8MchSSQ', attr='Mapbox')

    return m

In [ ]:
# Font
font_size = 16
fz = 1.5

plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['CMU Serif Roman'] + plt.rcParams['font.serif']
plt.rcParams['font.size'] = 16

mapply.init(
    n_workers=37,
    chunk_size=1,
    progressbar=True
)

In [ ]:
import os
WORKING_DIR = os.environ.get("REPO_ROOT", os.path.abspath(".."))  # repo root (notebooks run from notebooks/)
DATA_DIR = f'{WORKING_DIR}/data'

## Load data

### France shape

In [ ]:
france_file = f'{DATA_DIR}/france_shape/france.geojson'

fd = open(france_file, 'r')
geojson_content = json.load(fd)
fd.close()

france_mainland = Shape(geojson_content['features'][0]['geometry'])
france_mainland = france_mainland.buffer(0)
# Changing to lat, lon
france_shape = Shapely_transform(lambda x, y: (y, x), france_mainland)
france_shape

#### Paris Shape

In [ ]:
paris_file = f'{DATA_DIR}/cities/Paris.geojson'

fd = open(paris_file, 'r')
geojson_content = json.load(fd)
fd.close()

paris_shape = Shape(geojson_content['features'][0]['geometry'])
paris_shape = paris_shape.buffer(0)
# Changing to lat, lon
paris_shape = Shapely_transform(lambda x, y: (y, x), paris_shape)
paris_shape

### Load communes

In [ ]:
df_communes = gpd.read_file(f'{DATA_DIR}/carto/2024/COMMUNE.shp')
df_communes.head(2)

In [ ]:
df_communes = df_communes[['INSEE_COM', 'geometry']].copy()
df_communes = df_communes.rename(columns={'INSEE_COM': 'insee'})
df_communes.set_index('insee', inplace=True)
df_communes.reset_index(inplace=True)

all_insee = list(df_communes['insee'])
all_insee_len = [len(x) for x in all_insee]
print(f'Insee codes have lengths: {set(all_insee_len)}')

print(f'Number of insees: {df_communes.shape[0]}')
df_communes.head(2)

#### reverse the geometry

In [ ]:
df_communes['geometry'] = df_communes.mapply(lambda row: 
                                            Shapely_transform(lambda x, y: (y, x), row['geometry']), axis=1)

### Spatial filter - Communes inside France 

In [ ]:
df_communes['inside'] = df_communes.mapply(lambda row: france_shape.contains(row['geometry'].centroid), axis=1)

n_communes = len(df_communes)
n_communes_inside = len(df_communes[df_communes['inside'] == True])
n_communes_outside = len(df_communes[df_communes['inside'] == False])

print(f'Number of communes: {n_communes}')
print(f'Number of communes inside: {n_communes_inside}')
print(f'Number of communes outside: {n_communes_outside}')

In [ ]:
df_communes = df_communes[df_communes['inside']].copy().reset_index(drop=True)
df_communes = df_communes[['insee', 'geometry']].copy()
df_communes.head(2)

#### Communes urbanization level

In [ ]:
def get_simplify_type(urbanization_level):
    if urbanization_level.startswith('rural'):
        return 'rural'
    elif urbanization_level == 'urbain dense':
        return 'urban'
    elif urbanization_level == 'urbain densité intermédiaire':
        return 'suburban'
    else:
        return 'NA'

In [ ]:
df_communes_types = pd.read_csv(f'{DATA_DIR}/carto/communes_urban_rural.csv')
df_communes_types.rename(columns={'type': 'urbanization_level'}, inplace=True)
df_communes_types['insee'] = df_communes_types['insee'].astype(str)
df_communes_types['insee'] = df_communes_types.apply(lambda row: '0'+row['insee'] if len(row['insee']) < 5 else row['insee'] , axis=1)
df_communes_types['urbanization_level'] = df_communes_types.apply(lambda row: get_simplify_type(row['urbanization_level']), axis=1)

print(f'Number of insees with type: {df_communes_types.shape[0]}')
df_communes_types.head(2)

In [ ]:
df_communes = df_communes.join(df_communes_types.set_index('insee'), on='insee')
print(f'Number of insees with type: {df_communes.shape[0]}')
df_communes.head(2)

### Save geometries

In [ ]:
# there were some changes in the communes between 2019 and 2024, but minor enough that can be ignored
df_communes.to_pickle(f'{DATA_DIR}/carto/communes.pkl') 

## Load Social economical indicators

In [ ]:
def fix_insee_code(insee):
    return f'{str(insee).zfill(5)}'

def fix_median_income(income):
    try: 
        return float(income.replace(' ', '').replace(',', '').replace('.', ''))
    except:
        return np.nan

### 2019

In [ ]:
df_social_economic_2019 = pd.read_csv(f'{DATA_DIR}/dossier/2019/dossier_complet.csv', sep=';')
df_social_economic_2019['CODGEO'] = df_social_economic_2019['CODGEO'].apply(fix_insee_code)
df_social_economic_2019['MED19'] = df_social_economic_2019['MED19'].apply(fix_median_income)
df_social_economic_2019.head(2)

In [ ]:
row = df_social_economic_2019[df_social_economic_2019['CODGEO']=='01002'].iloc[0]

# P19_ACT1564: Nombre de personnes actives de 15 à 64 ans en 2019
# P19_ACTOCC1564: Nombre de personnes actives occupées de 15 à 64 ans en 2019
# P19_CHOMEUR1564: Nombre de personnes chômeuses de 15 à 64 ans en 2019
# P19_POP1564: Nombre de personnes de 15 à 64 ans en 2019
# Unemployed rate = P19_CHOMEUR1564 / P19_POP1564
# Employment rate = P19_ACTOCC1564 / P19_POP1564

print(row['P19_ACTOCC1564'], row['P19_CHOMEUR1564'], row['P19_POP1564'], row['P19_ACTOCC1564']/row['P19_POP1564'])

In [ ]:
economic_columns = [
                        'MED19', # Median income
                   ]

population_columns = [
                        'P19_POP', # Total population in 2019
                        'P19_POP0014', # Number of people aged 0 to 14 years old
                        'P19_POP1529', # Number of people aged 15 to 29 years old
                        'P19_POP3044', # Number of people aged 30 to 44 years old
                        'P19_POP4559', # Number of people aged 45 to 59 years old
                        'P19_POP6074', # Number of people aged 60 to 74 years old
                        'P19_POP7589', # Number of people aged 75 to 89 years old
                        'P19_POP90P' # Number of people aged 90 years old and more
                    ]

employment_columns = [
                        'P19_POP1564', # Number of people aged 15 to 64 years old
                        'P19_ACT1564', # Number of people actives aged 15 to 64 years old
                        'P19_ACTOCC1564', # Number of people actives occupied aged 15 to 64 years old
                        'P19_CHOMEUR1564', # Number of people unemployed aged 15 to 64 years old
                    ]


#                         'P19_SCOL0610', # Number of people in school from 6 to 10 years old
#                         'P19_SCOL1114', # Number of people in school from 11 to 14 years old
#                         'P19_SCOL1517', # Number of people in school from 15 to 17 years old
#                         'P19_SCOL1824', # Number of people in school from 18 to 24 years old
#                         'P19_SCOL2529', # Number of people in school from 25 to 29 years old
#                         'P19_SCOL30P', # Number of people in school from 30 years old and more
#                         'P19_NSCOL15P', # Number of out-of-school people aged 15 or over
#                         'P19_POP0205', # Number of people aged 2 to 5 years old
#                         'P19_POP0610', # Number of people aged 6 to 10 years old
#                         'P19_POP1114', # Number of people aged 11 to 14 years old
#                         'P19_POP1517', # Number of people aged 15 to 17 years old
#                         'P19_POP1824', # Number of people aged 18 to 24 years old
#                         'P19_POP2529', # Number of people aged 25 to 29 years old
#                         'P19_POP30P', # Number of people aged 30 years old and more
#                     ]

df_social_economic_2019 = df_social_economic_2019[['CODGEO'] + economic_columns + population_columns + employment_columns]


raname_columns = {
    'CODGEO': 'insee',
    'MED19': 'median_income',
    'P19_POP': 'pop',
    'P19_POP0014': 'pop_0_14',
    'P19_POP1529': 'pop_15_29',
    'P19_POP3044': 'pop_30_44',
    'P19_POP4559': 'pop_45_59',
    'P19_POP6074': 'pop_60_74',
    'P19_POP7589': 'pop_75_89',
    'P19_POP90P': 'pop_90',
    'P19_POP1564': 'pop_15_64',
    'P19_ACT1564': 'act_15_64',
    'P19_ACTOCC1564': 'act_ocupied_15_64',
    'P19_CHOMEUR1564': 'unemployed_15_64',
}


pop_range_columns = ['pop_0_14', 'pop_15_29', 'pop_30_44', 'pop_45_59', 'pop_60_74', 'pop_75_89', 'pop_90']
population_columns = ['pop'] + pop_range_columns


df_social_economic_2019 = df_social_economic_2019.rename(columns=raname_columns)
df_social_economic_2019.head(2)

#### Population ratio

In [ ]:
def pop_range_ratio(row):
    pop = row['pop']
    if pop == 0:
        return [0]*7
    
    pop_range_ratio_values = []

    for pop_range in pop_range_columns:
        ratio = row[pop_range] / pop
        pop_range_ratio_values.append(ratio)
    return pop_range_ratio_values

In [ ]:
df_social_economic_2019[pop_range_columns] = df_social_economic_2019.mapply(pop_range_ratio, axis=1, result_type='expand')
# keep population to compute population density and RCA
df_social_economic_2019.head(2)

#### Unemployment ratio

In [ ]:
df_social_economic_2019['unemployment_ratio'] = df_social_economic_2019['unemployed_15_64'] / df_social_economic_2019['pop_15_64']
df_social_economic_2019.drop(columns=['pop_15_64', 'act_15_64', 'act_ocupied_15_64', 'unemployed_15_64'], inplace=True)
df_social_economic_2019 = df_social_economic_2019[['insee', 'median_income', 'unemployment_ratio'] + list(df_social_economic_2019.columns[2:-1])]
df_social_economic_2019.head(2)

#### Save data

In [ ]:
df_social_economic_2019.to_pickle(f'{DATA_DIR}/dossier/df_social_economical_2019.pkl')

### 2024

In [ ]:
# this is the most recent data available
df_social_economic_2024 = pd.read_csv(f'{DATA_DIR}/dossier/2023/dossier_complet.csv', sep=';')
df_social_economic_2024['CODGEO'] = df_social_economic_2024['CODGEO'].apply(fix_insee_code)
df_social_economic_2024['MED21'] = df_social_economic_2024['MED21'].apply(fix_median_income)
df_social_economic_2024.head(2)

In [ ]:
economic_columns = [
                        'MED21', # Median income
                   ]

population_columns = [
                        'P21_POP', # Total population in 2021
                        'P21_POP0014', # Number of people aged 0 to 14 years old
                        'P21_POP1529', # Number of people aged 15 to 29 years old
                        'P21_POP3044', # Number of people aged 30 to 44 years old
                        'P21_POP4559', # Number of people aged 45 to 59 years old
                        'P21_POP6074', # Number of people aged 60 to 74 years old
                        'P21_POP7589', # Number of people aged 75 to 89 years old
                        'P21_POP90P' # Number of people aged 90 years old and more
                    ]

employment_columns = [
                        'P21_POP1564', # Number of people aged 15 to 64 years old
                        'P21_ACT1564', # Number of people actives aged 15 to 64 years old
                        'P21_ACTOCC1564', # Number of people actives occupied aged 15 to 64 years old
                        'P21_CHOM1564', # Number of people unemployed aged 15 to 64 years old
                    ]


#                         'P21_SCOL0205', # Number of people in school from 2 to 5 years old 
#                         'P21_SCOL0610', # Number of people in school from 6 to 10 years old
#                         'P21_SCOL1114', # Number of people in school from 11 to 14 years old
#                         'P21_SCOL1517', # Number of people in school from 15 to 17 years old
#                         'P21_SCOL1824', # Number of people in school from 18 to 24 years old
#                         'P21_SCOL2529', # Number of people in school from 25 to 29 years old
#                         'P21_SCOL30P', # Number of people in school from 30 years old and more
#                         'P21_NSCOL15P', # Number of out-of-school people aged 15 or over
#                         'P21_POP0205', # Number of people aged 2 to 5 years old
#                         'P21_POP0610', # Number of people aged 6 to 10 years old
#                         'P21_POP1114', # Number of people aged 11 to 14 years old
#                         'P21_POP1517', # Number of people aged 15 to 17 years old
#                         'P21_POP1824', # Number of people aged 18 to 24 years old
#                         'P21_POP2529', # Number of people aged 25 to 29 years old
#                         'P21_POP30P', # Number of people aged 30 years old and more
#                     ]

df_social_economic_2024 = df_social_economic_2024[['CODGEO'] + economic_columns + population_columns + employment_columns]


raname_columns = {
    'CODGEO': 'insee',
    'MED21': 'median_income',
    'P21_POP': 'pop',
    'P21_POP0014': 'pop_0_14',
    'P21_POP1529': 'pop_15_29',
    'P21_POP3044': 'pop_30_44',
    'P21_POP4559': 'pop_45_59',
    'P21_POP6074': 'pop_60_74',
    'P21_POP7589': 'pop_75_89',
    'P21_POP90P': 'pop_90',
    'P21_POP1564': 'pop_15_64',
    'P21_ACT1564': 'act_15_64',
    'P21_ACTOCC1564': 'act_ocupied_15_64',
    'P21_CHOM1564': 'unemployed_15_64',
}

df_social_economic_2024 = df_social_economic_2024.rename(columns=raname_columns)
df_social_economic_2024.head(2)

#### Population ratio

In [ ]:
df_social_economic_2024[pop_range_columns] = df_social_economic_2024.mapply(pop_range_ratio, axis=1, result_type='expand')
# keep population to compute population density and RCA
df_social_economic_2024.head(2)

#### Unemployment ratio

In [ ]:
df_social_economic_2024['unemployment_ratio'] = df_social_economic_2024['unemployed_15_64'] / df_social_economic_2024['pop_15_64']
df_social_economic_2024.drop(columns=['pop_15_64', 'act_15_64', 'act_ocupied_15_64', 'unemployed_15_64'], inplace=True)
df_social_economic_2024 = df_social_economic_2024[['insee', 'median_income', 'unemployment_ratio'] + list(df_social_economic_2024.columns[2:-1])]
df_social_economic_2024.head(2)

#### Save data

In [ ]:
df_social_economic_2024.to_pickle(f'{DATA_DIR}/dossier/df_social_economical_2024.pkl')

## Compare social economical indicators

In [ ]:
df_income_2019 = df_social_economic_2019[['insee', 'median_income']].copy()
df_income_2019 = df_income_2019.rename(columns={'median_income': 'median_income_2019'})

df_income_2024 = df_social_economic_2024[['insee', 'median_income']].copy()
df_income_2024 = df_income_2024.rename(columns={'median_income': 'median_income_2024'})

df_income = df_income_2019.merge(df_income_2024, on='insee', how='inner')

In [ ]:
plt.plot(df_income['median_income_2019'], df_income['median_income_2024'], 'o', alpha=0.25)
plt.plot([0, 60000], [0, 60000], 'r--')
plt.xlabel('Median income 2019')
plt.ylabel('Median income 2024 (2021)')
plt.xlim([0, 60000])
plt.ylim([0, 60000])
plt.show()

## Example

#### Selected the ones in Paris

In [ ]:
df_communes['inside_paris'] = df_communes.mapply(lambda row: row['geometry'].intersects(paris_shape), axis=1)
df_communes_paris = df_communes[df_communes['inside_paris']].copy()

In [ ]:
df_income_2019 = df_social_economic_2019[['insee', 'median_income']]
df_income_2024 = df_social_economic_2024[['insee', 'median_income']]
df_communes_paris_income_2019 = df_communes_paris.merge(df_income_2019, on='insee', how='left')
df_communes_paris_income_2024 = df_communes_paris.merge(df_income_2024, on='insee', how='left')

df_communes_paris_income_2019.shape, df_communes_paris_income_2024.shape

In [ ]:
m = get_map(zoom_start=6)

my_cmap = cm.get_cmap('RdYlGn')
my_norm = colrs.Normalize(vmin = 10000, vmax=50000)


for row in df_communes_paris_income_2019.to_dict(orient='records'):
    commune_shape = row['geometry']
    insee = row['insee']
    income = row['median_income']

    color = colrs.to_hex(my_cmap(my_norm(income)))
    
    folium.Polygon(mapping(commune_shape)['coordinates'][0],
        color=color,
        fill_color=color,
        opacity = .85,
        fill_opacity = 0.5,
        weight = 1,
        tooltip=f'{income}'
    ).add_to(m)

m